<a href="https://colab.research.google.com/github/kevinketerlondon-tech/Data-Analyst-portfolio/blob/main/00b_build_location_grouped_split.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 00b: Build the location-grouped train / validation / test split
**Kevin Keter · University of East London**

Images closer than 100 m to each other are linked into location groups; every group goes into exactly one split
(about 70 / 15 / 15 %), balanced for study area, vegetation cover and camera type. Output: `split_v3_grouped.csv`.
Run time about 1 minute, no GPU needed.

## 1. Settings

In [ ]:
import os, glob, zipfile
import pandas as pd
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/Dissertation/revision"
WORK = "/content/work"; os.makedirs(WORK, exist_ok=True)
class Args: pass
args = Args()
# Roboflow COCO-segmentation export (Version 3, full resolution), unzipped once
zips = [z for z in glob.glob(f"{DRIVE}/*.zip") if "v3" in os.path.basename(z).lower()]
assert zips, "Put the Roboflow v3 COCO-segmentation zip in " + DRIVE
args.coco_dir = f"{WORK}/roboflow_v3"
if not os.path.exists(f"{args.coco_dir}/train"):
    zipfile.ZipFile(zips[0]).extractall(args.coco_dir)
print("Dataset:", args.coco_dir)
args.metadata = f"{DRIVE}/mapillary_metadata_312.csv"
args.link_m, args.iterations = 100.0, 200000
# If the file already exists (it was used for the reported results), write a second copy and compare,
# so re-running this notebook never overwrites the original.
MAIN = f"{DRIVE}/split_v3_grouped.csv"
args.out = MAIN if not os.path.exists(MAIN) else MAIN.replace(".csv", "_rerun.csv")
print("Output:", args.out)

Mounted at /content/drive
Dataset: /content/work/roboflow_v3
Output: /content/drive/MyDrive/Dissertation/revision/split_v3_grouped_rerun.csv


## 2. Run

In [ ]:
import json, os
import numpy as np, pandas as pd
from PIL import Image, ImageDraw
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components


d = pd.read_csv(args.metadata, dtype={"image_id": str, "sequence": str})
lat, lng = np.radians(d.lat.values), np.radians(d.lng.values)
a = np.sin((lat[None] - lat[:, None]) / 2) ** 2 + np.cos(lat[:, None]) * np.cos(lat[None]) * np.sin((lng[None] - lng[:, None]) / 2) ** 2
D = 6371000 * 2 * np.arcsin(np.sqrt(a))
k, lab = connected_components(csr_matrix((D < args.link_m).astype(int)), directed=False)
d["group"] = lab
d["borough"] = np.where(d.lng > -0.07, "newham", "southwark")       # dissertation study-box labels (east / central box)

vf, fn = {}, {}
for s in ["train", "valid"]:
    c = json.load(open(os.path.join(args.coco_dir, s, "_annotations.coco.json"))); by = {}
    for an in c["annotations"]: by.setdefault(an["image_id"], []).append(an)
    for im in c["images"]:
        m = Image.new("L", (im["width"], im["height"]), 0); dr = ImageDraw.Draw(m)
        for an in by.get(im["id"], []):
            for seg in an.get("segmentation", []):
                if len(seg) >= 6: dr.polygon([(seg[i], seg[i + 1]) for i in range(0, len(seg), 2)], fill=1)
        iid = im["file_name"].split("_jpg")[0]; vf[iid] = np.asarray(m).mean(); fn[iid] = f"{s}/{im['file_name']}"
d["veg_frac"] = d.image_id.map(vf); d["rf_file"] = d.image_id.map(fn)
assert d.veg_frac.notna().all(), "metadata contains images that are not in the COCO export"

gi = d.group.values
M = np.stack([np.bincount(gi, minlength=k), np.bincount(gi, (d.borough == "newham").values, k),
              np.bincount(gi, d.veg_frac.values, k), np.bincount(gi, (d.camera_type != "perspective").values, k)], 1)
tot, target = M.sum(0), np.array([.70, .15, .15])
rng = np.random.default_rng(42)
A = rng.choice(3, size=(args.iterations, k), p=target)
cost = np.zeros(args.iterations)
for s in range(3):
    S = (A == s).astype(float) @ M; ns = np.maximum(S[:, 0], 1)
    cost += (np.abs(S[:, 0] / tot[0] - target[s]) * 10 + np.abs(S[:, 1] / ns - tot[1] / tot[0]) * 3
             + np.abs(S[:, 2] / ns - tot[2] / tot[0]) * 10 + np.abs(S[:, 3] / ns - tot[3] / tot[0]) * 3)
    cost += ((A == s).sum(1) < 5) * 99
best = A[cost.argmin()]
d["new_split"] = np.array(["train", "val", "test"])[best[gi]]
d.to_csv(args.out, index=False)

for s in ["train", "val", "test"]:
    x = d[d.new_split == s]
    print(f"{s:5s} n={len(x):3d} groups={x.group.nunique():2d} east_box={(x.borough == 'newham').mean():.0%} "
          f"veg={x.veg_frac.mean():.3f} 360/fisheye={(x.camera_type != 'perspective').mean():.0%}")
idx = {s: np.where(d.new_split == s)[0] for s in ["train", "val", "test"]}
for p, q in [("train", "test"), ("train", "val"), ("val", "test")]:
    print(f"minimum distance {p}-{q}: {D[np.ix_(idx[p], idx[q])].min():.0f} m")

train n=217 groups=37 east_box=42% veg=0.109 360/fisheye=18%
val   n= 47 groups= 9 east_box=43% veg=0.121 360/fisheye=13%
test  n= 46 groups= 9 east_box=46% veg=0.125 360/fisheye=15%
minimum distance train-test: 129 m
minimum distance train-val: 104 m
minimum distance val-test: 405 m


### Check against the file used in the paper

In [ ]:
if args.out != MAIN:
    a, b = pd.read_csv(MAIN, dtype=str), pd.read_csv(args.out, dtype=str)
    same = (a.set_index("image_id").new_split == b.set_index("image_id").new_split).all()
    print("Identical to the split used in the paper:", bool(same))

Identical to the split used in the paper: True
